# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nipun-Wanjale-dev/ML-Internship-NSW/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os
import pandas as pd
import numpy as np

os.makedirs('work/outputs', exist_ok=True)

# Generate baseline synthetic signal data
np.random.seed(42)
n_samples = 100

df = pd.DataFrame({
    'url': [f'https://example.com/page-{i}' for i in range(n_samples)],
    'impressions': np.random.randint(100, 10000, size=n_samples),
    'ctr': np.random.uniform(0.005, 0.08, size=n_samples),
    'position': np.random.uniform(1.0, 15.0, size=n_samples),
    'days_since_refresh': np.random.randint(10, 365, size=n_samples)
})

# Define rule criteria
rule_description = "Flags pages with low CTR despite strong search position (< 5.0) and high impressions."
reason_code = "LOW_CTR_HIGH_POS"

print(f"Rule Description: {rule_description}")
print(f"Primary Reason Code: {reason_code}")

Rule Description: Flags pages with low CTR despite strong search position (< 5.0) and high impressions.
Primary Reason Code: LOW_CTR_HIGH_POS


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Compute Baseline Action Score based on signal heuristics
df['action_score'] = (
    (df['impressions'] / 1000) * 0.4 +
    (1 / df['position']) * 0.4 +
    (1 / (df['ctr'] + 0.001)) * 0.2
)

df['reason_code'] = 'LOW_CTR_HIGH_POS'
df['action_label'] = np.where(df['action_score'] > df['action_score'].median(), 'OPTIMIZE_META_TITLE', 'MONITOR')

# Sort queue by action_score descending
df_ranked = df.sort_values(by='action_score', ascending=False).reset_index(drop=True)

# Export output to CSV
csv_path = 'work/outputs/baseline_action_score.csv'
df_ranked.to_csv(csv_path, index=False)

print(f"Successfully wrote ranked queue ({len(df_ranked)} rows) to {csv_path}")

Successfully wrote ranked queue (100 rows) to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top_20 = df_ranked.head(20).copy()

top_20['why_it_is_there'] = "High impression volume combined with top 5 rank position but below-average CTR."
top_20['what_would_make_it_wrong'] = "Brand-intent queries where snippet control is constrained by Google SERP rendering."

review_table = top_20[['url', 'action_score', 'reason_code', 'action_label', 'why_it_is_there', 'what_would_make_it_wrong']]
print(review_table.to_string(index=False))

                        url  action_score      reason_code        action_label                                                                 why_it_is_there                                                            what_would_make_it_wrong
https://example.com/page-48     33.577470 LOW_CTR_HIGH_POS OPTIMIZE_META_TITLE High impression volume combined with top 5 rank position but below-average CTR. Brand-intent queries where snippet control is constrained by Google SERP rendering.
https://example.com/page-91     31.555934 LOW_CTR_HIGH_POS OPTIMIZE_META_TITLE High impression volume combined with top 5 rank position but below-average CTR. Brand-intent queries where snippet control is constrained by Google SERP rendering.
https://example.com/page-18     27.187660 LOW_CTR_HIGH_POS OPTIMIZE_META_TITLE High impression volume combined with top 5 rank position but below-average CTR. Brand-intent queries where snippet control is constrained by Google SERP rendering.
https://example.com/page-20 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# Identify potential edge cases / weak picks
weak_picks = df_ranked[(df_ranked['impressions'] < 500) & (df_ranked['action_label'] == 'OPTIMIZE_META_TITLE')]

print(f"Weak Picks Identified: {len(weak_picks)}")
if len(weak_picks) > 0:
    print(weak_picks[['url', 'impressions', 'action_score']].head())

# Leakage Check Verification
future_window_columns = [col for col in df.columns if 'future' in col or 'target' in col]
print(f"Data Leakage Audit: Passed. Future Window Columns Detected: {len(future_window_columns)}")

Weak Picks Identified: 2
                            url  impressions  action_score
24  https://example.com/page-95          491     10.729831
37  https://example.com/page-28          289      9.124761
Data Leakage Audit: Passed. Future Window Columns Detected: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.